
#Bronze Ingestion Framework

##Objective -
- load source files from azure container
- enforce schema contarcts
- add bronze audit metadata columns
- write to bronze delta tables
- register tables in unity catalog

## Parameter configuration

In [0]:
# -------------------------------------
# environment configuration
# -------------------------------------

catalog_name = "cdl"
bronze_schema_name = "bronze"

storage_account_name = "datalakedevesh"

source_container_name = "source-cdl"
destination_container_name = "destination-cdl"

# unity catalog source volume paths

landing_volume_path = f"/Volumes/{catalog_name}/{bronze_schema_name}/landing_files/"

# direct cloud paths
source_base_path = f"abfss://{source_container_name}@{storage_account_name}.dfs.core.windows.net/"
bronze_base_path = f"abfss://{destination_container_name}@{storage_account_name}.dfs.core.windows.net/bronze/"

print("Catalog:",catalog_name)
print("Bronze Schema:",bronze_schema_name)
print("Landing volume path:",landing_volume_path)
print("Source cloud path:",source_base_path)
print("Destination cloud path:",bronze_base_path)



In [0]:
# spark.sql(f"show schemas in {catalog_name}").display()
# spark.sql(f"show volumes in {catalog_name}.{bronze_schema_name}").display()

In [0]:
# display(dbutils.fs.ls(landing_volume_path))

##Source files Discovery

In [0]:
source_files = [file 
                for file in (dbutils.fs.ls(landing_volume_path)) 
                    if file.name.endswith(".json")]

display(source_files)

In [0]:
# s2 = []

# for file in (dbutils.fs.ls(landing_volume_path)):
#     s2.append(file)

# display(s2[0])

##Creation of a file parser to separate and file name and file date

In [0]:
import re

def parse_source_file_name(file_name):
    pattern = r"^(.*)_(\d{4}_\d{2}_\d{2})\.json$"
    match = re.match(pattern, file_name)
    if not match:
        raise ValueError(f"Invalid file name pattern: {file_name}")
    entity_name = match.group(1)
    date = match.group(2).replace("_", "-")
    return entity_name, date

# a,b = parse_source_file_name("call_activity_2026_06_15.json")
# print(f"entity is {a} and date is {b}")

## Metadata config for driving ingestiong using source information, avoiding harcdoing and enterprise practices

In [0]:
# print(type(source_files[0]))

# print(source_files[0])

# for file in source_files:
#     print(type(file))
#     print(file)
#     print(file.path)
#     # display(file)
#     break

In [0]:
# metadata creation for ingestion fw

discovered_files = []

for file in source_files:
    entity_name, date = parse_source_file_name(file.name)
    discovered_files.append({
        "source_file_path": file.path,
        "source_file_name": file.name,
        "source_entity": entity_name,
        "source_file_date": date,
        "file_format": "json"
    })
display(discovered_files)

In [0]:
# spark.read.text(discovered_files[0]["source_file_path"]).display()

In [0]:
# [file["source_entity"] for file in discovered_files]
# print(discovered_files[0]["source_file_path"])

In [0]:
# print(discovered_files)

# df_sample = spark.read.json(discovered_files[0]["source_file_path"])
# df_sample.printSchema()

##Defining data contracts

In [0]:
# spark.read.text(source_files[0].path).display()

In [0]:
# schema contract dictionary

schema_contracts = {
    "call_activity":[
        {"column_name":"call_id","data_type":"string","nullable":False},
        {"column_name":"call_date","data_type":"date","nullable":False},
        {"column_name":"call_status","data_type":"string","nullable":False},
        {"column_name":"call_type","data_type":"string","nullable":False},
        {"column_name":"duration_minutes","data_type":"long","nullable":False},
        {"column_name":"hcp_id","data_type":"string","nullable":False},
        {"column_name":"product_id","data_type":"string","nullable":False},
        {"column_name":"source_file_date","data_type":"date","nullable":False},
        {"column_name":"source_system","data_type":"string","nullable":False},
        {"column_name":"territory_id","data_type":"string","nullable":False}
    ],
    "hcp_master":[
        {"column_name": "city", "data_type": "string", "nullable": False},
        {"column_name": "effective_date", "data_type": "date", "nullable": False},
        {"column_name": "hcp_id", "data_type": "string", "nullable": False},
        {"column_name": "hcp_name", "data_type": "string", "nullable": False},
        {"column_name": "is_active", "data_type": "boolean", "nullable": False},
        {"column_name": "source_file_date", "data_type": "date", "nullable": False},
        {"column_name": "specialty", "data_type": "string", "nullable": False},
        {"column_name": "state", "data_type": "string", "nullable": False},
        {"column_name": "territory_id", "data_type": "string", "nullable": False}   
    ],
    "product_master":[
        {"column_name": "brand", "data_type": "string", "nullable": False},
        {"column_name": "effective_date", "data_type": "date", "nullable": False},
        {"column_name": "is_active", "data_type": "boolean", "nullable": False},
        {"column_name": "launch_year", "data_type": "long", "nullable": False},
        {"column_name": "product_id", "data_type": "string", "nullable": False},
        {"column_name": "product_name", "data_type": "string", "nullable": False},
        {"column_name": "source_file_date", "data_type": "date", "nullable": False},
        {"column_name": "therapy_area", "data_type": "string", "nullable": False}
    ],
    "rx_transactions":[
        {"column_name": "hcp_id", "data_type": "string", "nullable": False},
        {"column_name": "prescription_count", "data_type": "long", "nullable": False},
        {"column_name": "product_id", "data_type": "string", "nullable": False},
        {"column_name": "rx_date", "data_type": "date", "nullable": False},
        {"column_name": "rx_id", "data_type": "string", "nullable": False},
        {"column_name": "source_file_date", "data_type": "date", "nullable": False},
        {"column_name": "source_system", "data_type": "string", "nullable": False},
        {"column_name": "territory_id", "data_type": "string", "nullable": False}
    ],
    "territory_master":[
        {"column_name": "effective_date", "data_type": "date", "nullable": False},
        {"column_name": "region", "data_type": "string", "nullable": False},
        {"column_name": "sales_rep_id", "data_type": "string", "nullable": False},
        {"column_name": "sales_rep_name", "data_type": "string", "nullable": False},
        {"column_name": "source_file_date", "data_type": "date", "nullable": False},
        {"column_name": "territory_id", "data_type": "string", "nullable": False},
        {"column_name": "territory_name", "data_type": "string", "nullable": False},
        {"column_name": "zone", "data_type": "string", "nullable": False}
    ]
}

# print(schema_contracts)
# print(schema_contracts["call_activity"][0]["column_name"])

##Converting schema contracts dictionary into spark schema

In [0]:
# simple function to convert the schema contarct into spark schema

from pyspark.sql.types import *

def build_spark_schema(entity_name):
    fields = []

    for column in schema_contracts[entity_name]:

        if column["data_type"] == "string":
            spark_data_type = StringType()

        elif column["data_type"] == "long":
            spark_data_type = LongType()

        elif column["data_type"] == "integer":
            spark_data_type = IntegerType()

        elif column["data_type"] == "double":
            spark_data_type = DoubleType()

        elif column["data_type"] == "boolean":
            spark_data_type = BooleanType()

        elif column["data_type"] == "date":
            spark_data_type = DateType()

        elif column["data_type"] == "timestamp":
            spark_data_type = TimestampType()
            
        else:
            raise ValueError(f"Invalid data type: {column['data_type']}")
        
        fields.append(StructField(column["column_name"], spark_data_type, column["nullable"]))
        # print(StructField(column["column_name"], spark_data_type, column["nullable"]))
    
    # print(fields)
    return StructType(fields)

call_activity_schema = build_spark_schema("call_activity")

# df_call_activity = spark.read.schema(call_activity_schema).json(discovered_files[0]["source_file_path"])
# df_call_activity.printSchema()
# df_call_activity.display()


##Bronze schema metadata column addition

In [0]:
# Addition of bronze audit columns

from pyspark.sql.functions import *

def add_bronze_metadata(df, file_metadata,ingestion_batch_id):
    df_with_audit = df.withColumn("bronze_load_ts",current_timestamp())\
                      .withColumn("source_file_name",lit(file_metadata["source_file_name"]))\
                      .withColumn("source_file_path",lit(file_metadata["source_file_path"]))\
                      .withColumn("ingestion_file_date",to_date(lit(file_metadata["source_file_date"])))\
                      .withColumn("ingestion_batch_id",lit(ingestion_batch_id))
    return df_with_audit


In [0]:
#testing

# import uuid

# ingestion_batch_id = str(uuid.uuid4())

# df_call_activity_2 = add_bronze_metadata(df_call_activity, discovered_files[0], ingestion_batch_id)
# df_call_activity_2.printSchema()
# df_call_activity_2.display()
# df_call_activity.display()

In [0]:
# # Derive target bronze table and path

# target_table_name = f"{catalog_name}.{bronze_schema_name}.{discovered_files[0]['source_entity']}_bronze"
# target_table_path = f"{bronze_base_path}{discovered_files[0]['source_entity']}_bronze"

# print(target_table_name)
# print(target_table_path)

# df_call_activity_2.write\
#     .format("delta")\
#     .mode("overwrite")\
#     .option("path",target_table_path)\
#     .saveAsTable(target_table_name)

# print("successfully written")

##External delta table write in azure container

In [0]:
def write_to_bronze(df, file_metadata,write_mode = "overwrite"):
    entity_name = file_metadata["source_entity"]
    target_table_name = f"{catalog_name}.{bronze_schema_name}.{entity_name}_bronze"
    target_table_path = f"{bronze_base_path}{entity_name}_bronze"

    print(f"writing entity: {entity_name}")
    print(f"target table: {target_table_name}")
    print(f"target path: {target_table_path}")
    print(f"write mode: {write_mode}")

    df.write.format("delta")\
        .mode(write_mode)\
        .option("path",target_table_path)\
        .saveAsTable(target_table_name)

    print(f"successfull written: {target_table_name}")

    return {
        "entity_name":entity_name,
        "target_table_name":target_table_name,
        "target_table_path":target_table_path,
        "write_mode":write_mode,
        "status":"success"
    }
#write_to_bronze(df_call_activity_2, discovered_files[0
    

In [0]:
# write_result = write_to_bronze(
#     df_call_activity_2,
#     discovered_files[0],
#     write_mode="overwrite"
# )

# write_result

In [0]:
# %sql
# SHOW EXTERNAL LOCATIONS;

In [0]:
# spark.sql("SELECT * FROM cdl.bronze.call_activity_bronze").display()
# spark.sql("DESCRIBE DETAIL cdl.bronze.call_activity_bronze").display()

## Function to ingest 1 file and subsequently call all associated functions in sequence

In [0]:
# caller function to process one source file
from datetime import datetime

def process_one_file(file_metadata, ingestion_batch_id,write_mode = "overwrite"):

    load_start_ts = datetime.now()
    entity_name = file_metadata["source_entity"]
    source_file_path = file_metadata["source_file_path"]
    source_file_name = file_metadata["source_file_name"]
    source_file_date = file_metadata["source_file_date"]
    target_table_name = f"{catalog_name}.{bronze_schema_name}.{entity_name}_bronze"
    target_table_path = f"{bronze_base_path}{entity_name}_bronze"

    print(f"started processing entity: {entity_name}")
    print(f"source file: {source_file_path}")

    try:
        # 1. Build schema from contract

        schema = build_spark_schema(entity_name)

        # 2. Read the json file using explicit schema

        df = spark.read.schema(schema).json(source_file_path)

        # 3. count rows before write
        
        row_count = df.count()

        # 4. Add bronze metadata

        df_with_metadata = add_bronze_metadata(df, file_metadata,ingestion_batch_id)

        # 5. Write to bronze table

        write_result = write_to_bronze(df_with_metadata, file_metadata,write_mode)

        load_end_ts = datetime.now()

        #6 Write success log

        log_record = {
            "ingestion_batch_id":ingestion_batch_id,
            "source_name":entity_name,
            "source_file_name":source_file_name,
            "source_file_path":source_file_path,
            "source_file_date":source_file_date,
            "target_table_name":target_table_name,
            "target_table_path":target_table_path,
            "write_mode":write_mode,
            "row_count":row_count,
            "load_start_ts":load_start_ts,
            "load_end_ts":load_end_ts,
            "load_duration":load_end_ts - load_start_ts,
            "log_status":"success",
            "error_message":None
        }

        write_ingestion_log(log_record)

        print(f"finished processing entity: {entity_name}")

        return log_record
    
    except Exception as e:

        load_end_ts = datetime.now()

        # write failure log
        log_record = {
            "ingestion_batch_id": ingestion_batch_id,
            "source_entity": entity_name,
            "source_file_name": source_file_name,
            "source_file_path": source_file_path,
            "source_file_date": source_file_date,
            "target_table_name": target_table_name,
            "target_table_path": target_table_path,
            "write_mode": write_mode,
            "row_count": 0,
            "load_status": "failed",
            "load_start_ts": load_start_ts,
            "load_end_ts": load_end_ts,
            "error_message": str(e)
        }

        write_ingestion_log(log_record)

        print(f"failed processing entity: {entity_name}")
        print(f"error: {str(e)}")

        return log_record

# test the caller function

# process_one_file(discovered_files[0], ingestion_batch_id)


## Ingestion log record helper function

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

from pyspark.sql.functions import to_date, col

def write_ingestion_log(log_record):
    
    log_schema = StructType([
        StructField("ingestion_batch_id", StringType(), True),
        StructField("source_entity", StringType(), True),
        StructField("source_file_name", StringType(), True),
        StructField("source_file_path", StringType(), True),
        StructField("source_file_date", StringType(), True),
        StructField("target_table_name", StringType(), True),
        StructField("target_table_path", StringType(), True),
        StructField("write_mode", StringType(), True),
        StructField("row_count", LongType(), True),
        StructField("load_status", StringType(), True),
        StructField("load_start_ts", TimestampType(), True),
        StructField("load_end_ts", TimestampType(), True),
        StructField("error_message", StringType(), True)
    ])
    
    log_df = spark.createDataFrame([log_record], schema=log_schema)
    
    log_df = log_df.withColumn(
        "source_file_date",
        to_date(col("source_file_date"))
    )
    
    (
        log_df.write
            .format("delta")
            .mode("append")
            .saveAsTable("cdl.audit.ingestion_load_log")
    )

##Batch ingestion for all the files present in metadata config

In [0]:
# process all the files togethor

import uuid

ingestion_batch_id = str(uuid.uuid4())

load_results = []

for file_metadata in discovered_files:
    # if file_metadata["source_entity"] == "call_activity":
        # print(file_metadata)
    load_result = process_one_file(file_metadata, ingestion_batch_id,write_mode = "overwrite")
    load_results.append(load_result)

load_results

In [0]:
# display(spark.createDataFrame(load_results))

In [0]:
# print(discovered_files[4]["source_file_name"])
# spark.read.format("json").load(discovered_files[4]["source_file_path"]).printSchema()
# # spark.read.format("json").load(discovered_files[2]["source_file_path"]).display()

In [0]:
spark.sql("DESCRIBE TABLE cdl.audit.ingestion_load_log").display()